# 01b — Portfolio Configuration Study: Sequential FCR-D Bidding (Perfect Forecasts)

Runs the corrected upper-bound model across **6 portfolio configurations** to test
sensitivity to portfolio size, power-to-energy ratio, heterogeneity, and battery efficiency.

**Model (v6):** SOC dynamics at T^max, υ_E^D = 1/3 h, LER NEM υ_P = 0.20, E-9a/b dropped.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, os, time
from datetime import date as dt_date

import pyomo.environ as pyo
from pyomo.environ import SolverFactory

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.max_open_warning': 0, 'figure.dpi': 120})

print('All imports OK')


## 1. Paths & shared constants

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
BASE_DIR       = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_OUT_DIR   = os.path.join(BASE_DIR, 'data', 'data_out')
FIGURES_DIR    = os.path.join(BASE_DIR, 'figures')
COMBINED_CSV   = os.path.join(DATA_OUT_DIR, 'combined_2025_with_frequency.csv')
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(DATA_OUT_DIR, exist_ok=True)

# ── Model constants (fixed across all configs) ───────────────────────────
SOC_INIT_FRAC  = 0.50
T_SUSTAIN      = 1/3       # regulatory min υ_E^D (Energinet, 2023)
UPSILON_P      = 0.20      # LER NEM cross-direction power factor
T_MAX          = 24
ORE_TO_DKK     = 1.0 / 100.0
SOLVER_NAME    = 'appsi_highs'

# ── Synthetic portfolio seed & type probabilities ────────────────────────
RNG_SEED       = 42
PROB_B_TYPE    = 0.60
SCALE_MU       = 1.0
SCALE_SIGMA    = 1.0

print('Constants defined')


## 2. Portfolio configurations

Each config is a dict with per-EC arrays for `b_max`, `s_max`, `eta`, plus `p_min`.

In [ ]:
def make_config(name, n_ec, b_max_per_ec, s_max_per_ec, eta_per_ec, p_min=100.0):
    "Build a portfolio config dict. Arrays or scalars accepted."
    b = np.full(n_ec, b_max_per_ec) if np.isscalar(b_max_per_ec) else np.array(b_max_per_ec)
    s = np.full(n_ec, s_max_per_ec) if np.isscalar(s_max_per_ec) else np.array(s_max_per_ec)
    e = np.full(n_ec, eta_per_ec)   if np.isscalar(eta_per_ec)   else np.array(eta_per_ec)
    return {
        'name': name, 'n_ec': n_ec,
        'b_max': b, 's_max': s, 'eta': e,
        'p_min': p_min,
        'p_max': float(b.sum()),
        's_total': float(s.sum()),
    }

# ── Config 1: Barely viable (stress test at min-bid floor) ───────────────
cfg1 = make_config('C1_barely_viable', n_ec=10,
    b_max_per_ec=10.0, s_max_per_ec=20.0, eta_per_ec=0.95, p_min=100.0)

# ── Config 2: Realistic heterogeneous (20 ECs, varied sizes) ─────────────
rng_cfg = np.random.default_rng(123)
n2 = 20
b2 = rng_cfg.uniform(50, 100, n2).round(0)
s2 = b2 * rng_cfg.uniform(1.5, 3.0, n2).round(1)   # C1.5 to C3 ratio
cfg2 = make_config('C2_heterogeneous', n_ec=n2,
    b_max_per_ec=b2, s_max_per_ec=s2, eta_per_ec=0.95, p_min=100.0)

# ── Config 3: Baseline (current study) ───────────────────────────────────
cfg3 = make_config('C3_baseline', n_ec=10,
    b_max_per_ec=100.0, s_max_per_ec=200.0, eta_per_ec=0.95, p_min=100.0)

# ── Config 4: High power, low energy (thin battery, C2) ──────────────────
cfg4 = make_config('C4_thin_battery', n_ec=10,
    b_max_per_ec=100.0, s_max_per_ec=100.0, eta_per_ec=0.95, p_min=100.0)

# ── Config 5: Large modern (fewer, bigger ECs) ───────────────────────────
cfg5 = make_config('C5_large_modern', n_ec=5,
    b_max_per_ec=200.0, s_max_per_ec=600.0, eta_per_ec=0.95, p_min=100.0)

# ── Config 6: Mixed efficiency (same as baseline, degraded fleet) ────────
eta6 = np.array([0.95, 0.95, 0.92, 0.92, 0.88, 0.88, 0.85, 0.85, 0.80, 0.80])
cfg6 = make_config('C6_mixed_efficiency', n_ec=10,
    b_max_per_ec=100.0, s_max_per_ec=200.0, eta_per_ec=eta6, p_min=100.0)

ALL_CONFIGS = [cfg1, cfg2, cfg3, cfg4, cfg5, cfg6]

for c in ALL_CONFIGS:
    print(f"{c['name']:25s}  {c['n_ec']:2d} ECs  "
          f"{c['p_max']:6.0f} kW / {c['s_total']:6.0f} kWh  "
          f"η=[{c['eta'].min():.2f}–{c['eta'].max():.2f}]  "
          f"p_min={c['p_min']:.0f}")


## 3. Load & prepare data

In [ ]:
# ── Load raw CSV ──────────────────────────────────────────────────────────
df_raw = pd.read_csv(COMBINED_CSV, parse_dates=['hour_utc'])
df_raw = df_raw.sort_values(['hour_utc', 'ec_id']).reset_index(drop=True)
print(f'Loaded {len(df_raw)} rows  |  ec_ids: {sorted(df_raw["ec_id"].unique())}  |  '
      f'{df_raw["hour_utc"].min()} → {df_raw["hour_utc"].max()}')

df_market = df_raw.groupby('hour_utc').first().reset_index()

price_cols_ore = [
    'buy_price_inkl_vat_ore_kwh', 'sell_price_inkl_vat_ore_kwh',
    'price_ore_kwh_fcr_d_upp__d_1_early', 'price_ore_kwh_fcr_d_upp__d_1_late',
    'price_ore_kwh_fcr_d_ned__d_1_early', 'price_ore_kwh_fcr_d_ned__d_1_late',
]
for col in price_cols_ore:
    df_market[col.replace('ore', 'dkk')] = df_market[col] * ORE_TO_DKK

COL_BUY       = 'buy_price_inkl_vat_dkk_kwh'
COL_SELL      = 'sell_price_inkl_vat_dkk_kwh'
COL_UP_EARLY  = 'price_dkk_kwh_fcr_d_upp__d_1_early'
COL_UP_LATE   = 'price_dkk_kwh_fcr_d_upp__d_1_late'
COL_DOWN_EARLY= 'price_dkk_kwh_fcr_d_ned__d_1_early'
COL_DOWN_LATE = 'price_dkk_kwh_fcr_d_ned__d_1_late'
COL_ACT_UP    = 'y_act_fcrd_up'
COL_ACT_DOWN  = 'y_act_fcrd_down'

df_market['buyback_up']   = df_market[[COL_UP_EARLY, COL_UP_LATE]].max(axis=1)
df_market['buyback_down'] = df_market[[COL_DOWN_EARLY, COL_DOWN_LATE]].max(axis=1)
df_market['y_acc_up']   = (df_market[COL_UP_EARLY]   > 0).astype(float)
df_market['y_acc_down'] = (df_market[COL_DOWN_EARLY] > 0).astype(float)
df_market['date'] = df_market['hour_utc'].dt.date

# EC base profiles
df_b = df_raw[df_raw['ec_id']=='b'][['hour_utc','consumption','pv_production_kwh']].copy()
df_b = df_b.rename(columns={'consumption':'load_b','pv_production_kwh':'pv_b'})
df_s = df_raw[df_raw['ec_id']=='s'][['hour_utc','consumption','pv_production_kwh']].copy()
df_s = df_s.rename(columns={'consumption':'load_s','pv_production_kwh':'pv_s'})
df_all = df_market.merge(df_b, on='hour_utc', how='left').merge(df_s, on='hour_utc', how='left')
for c in ['load_b','pv_b','load_s','pv_s']:
    df_all[c] = df_all[c].fillna(0.0)

print(f'Combined data: {len(df_all)} rows')


## 4. Portfolio construction

In [ ]:
def generate_portfolio(n_ec, prob_b, scale_mu, scale_sigma, seed):
    rng = np.random.default_rng(seed)
    ec_types, scale_factors = [], []
    for _ in range(n_ec):
        ec_types.append('b' if rng.random() < prob_b else 's')
        while True:
            s = rng.normal(scale_mu, scale_sigma)
            if s > 0:
                scale_factors.append(s)
                break
    return ec_types, np.array(scale_factors)

def get_ec_profiles(df_day, ec_types, scale_factors):
    n_ec = len(ec_types)
    loads = np.zeros((n_ec, 24))
    pvs   = np.zeros((n_ec, 24))
    load_b, pv_b = df_day['load_b'].values, df_day['pv_b'].values
    load_s, pv_s = df_day['load_s'].values, df_day['pv_s'].values
    for e in range(n_ec):
        if ec_types[e] == 'b':
            loads[e] = load_b * scale_factors[e]
            pvs[e]   = pv_b   * scale_factors[e]
        else:
            loads[e] = load_s * scale_factors[e]
            pvs[e]   = pv_s   * scale_factors[e]
    return loads, pvs

print('Portfolio functions defined')


## 5. MILP models (parameterised per-EC)

Models accept per-EC arrays for `b_max`, `s_max`, `eta` to support heterogeneous portfolios.

In [ ]:
def build_early_model(loads, pvs, buy_price, sell_price,
                      fcrd_up_price, fcrd_down_price,
                      act_frac_up, act_frac_down,
                      y_acc_up, y_acc_down,
                      b_max_arr, s_max_arr, eta_arr, p_min, p_max):
    n_ec = loads.shape[0]
    E = range(n_ec); T = range(T_MAX)
    m = pyo.ConcreteModel('Early')
    m.E = pyo.Set(initialize=E); m.T = pyo.Set(initialize=T)

    m.p_im       = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_ex       = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.b_ch       = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.b_dis      = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.SOC        = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_up_res   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_down_res = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_up_act   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_down_act = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_net      = pyo.Var(m.E, m.T, within=pyo.Reals)
    m.z_up       = pyo.Var(m.T, within=pyo.Binary)
    m.z_down     = pyo.Var(m.T, within=pyo.Binary)

    # (E-obj)
    def obj_rule(m):
        return sum(
            buy_price[t]  * (sum(m.p_im[e,t] for e in E) - sum(m.p_down_act[e,t] for e in E))
          - sell_price[t] * (sum(m.p_ex[e,t] for e in E) - sum(m.p_up_act[e,t]   for e in E))
          - fcrd_up_price[t]   * sum(m.p_up_res[e,t]   for e in E)
          - fcrd_down_price[t] * sum(m.p_down_res[e,t] for e in E)
          for t in T)
    m.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

    # (E-1) p_net = p_im - p_ex
    m.c_net = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_net[e,t] - m.p_im[e,t] + m.p_ex[e,t] == 0)
    # (E-4) Battery power balance
    m.c_pb = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_net[e,t] + pvs[e,t] - loads[e,t]
            - (m.b_ch[e,t] + m.p_down_act[e,t])
            + (m.b_dis[e,t] + m.p_up_act[e,t]) == 0)
    # (E-5a,b) Activation tracking
    m.c_act_up = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_up_act[e,t] == y_acc_up[t] * act_frac_up[t] * m.p_up_res[e,t])
    m.c_act_dn = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_down_act[e,t] == y_acc_down[t] * act_frac_down[t] * m.p_down_res[e,t])

    # (E-6b) SOC at t=0
    m.c_soc0 = pyo.Constraint(m.E,
        rule=lambda m,e: m.SOC[e,0] == SOC_INIT_FRAC * s_max_arr[e]
            + eta_arr[e] * (m.b_ch[e,0] + m.p_down_act[e,0])
            - (1.0/eta_arr[e]) * (m.b_dis[e,0] + m.p_up_act[e,0]))
    # (E-6a) SOC dynamics t=1..T_MAX-1 (includes terminal hour)
    def soc_dyn(m, e, t):
        if t == 0:
            return pyo.Constraint.Skip
        return m.SOC[e,t] == m.SOC[e,t-1] \
            + eta_arr[e] * (m.b_ch[e,t] + m.p_down_act[e,t]) \
            - (1.0/eta_arr[e]) * (m.b_dis[e,t] + m.p_up_act[e,t])
    m.c_soc_dyn = pyo.Constraint(m.E, m.T, rule=soc_dyn)
    # (E-6c) Terminal SOC
    m.c_soc_end = pyo.Constraint(m.E,
        rule=lambda m,e: m.SOC[e, T_MAX-1] == SOC_INIT_FRAC * s_max_arr[e])

    # (E-7a) Sustain up
    m.c_sus_up = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.SOC[e,t] >= y_acc_up[t] * T_SUSTAIN / eta_arr[e] * m.p_up_res[e,t])
    # (E-7b) Sustain down
    m.c_sus_dn = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: s_max_arr[e] - m.SOC[e,t] >= y_acc_down[t] * T_SUSTAIN * eta_arr[e] * m.p_down_res[e,t])

    # (E-8a) LER power discharge
    m.c_dis_lim = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.b_dis[e,t] + m.p_up_res[e,t] + UPSILON_P * m.p_down_res[e,t] <= b_max_arr[e])
    # (E-8b) LER power charge
    m.c_ch_lim = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.b_ch[e,t] + m.p_down_res[e,t] + UPSILON_P * m.p_up_res[e,t] <= b_max_arr[e])

    # SOC upper bound
    m.c_soc_cap = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.SOC[e,t] <= s_max_arr[e])

    # (E-10) Min bid
    m.c_bid_up_lo = pyo.Constraint(m.T,
        rule=lambda m,t: p_min * m.z_up[t] <= sum(m.p_up_res[e,t] for e in E))
    m.c_bid_up_hi = pyo.Constraint(m.T,
        rule=lambda m,t: sum(m.p_up_res[e,t] for e in E) <= p_max * m.z_up[t])
    m.c_bid_dn_lo = pyo.Constraint(m.T,
        rule=lambda m,t: p_min * m.z_down[t] <= sum(m.p_down_res[e,t] for e in E))
    m.c_bid_dn_hi = pyo.Constraint(m.T,
        rule=lambda m,t: sum(m.p_down_res[e,t] for e in E) <= p_max * m.z_down[t])
    return m

print('build_early_model() defined')


In [ ]:
def build_late_model(loads, pvs, buy_price, sell_price,
                     fcrd_up_early_price, fcrd_down_early_price,
                     fcrd_up_late_price, fcrd_down_late_price,
                     buyback_up_price, buyback_down_price,
                     act_frac_up, act_frac_down,
                     y_acc_up, y_acc_down,
                     early_up_acc, early_down_acc,
                     b_max_arr, s_max_arr, eta_arr, p_min, p_max):
    n_ec = loads.shape[0]
    E = range(n_ec); T = range(T_MAX)
    m = pyo.ConcreteModel('Late')
    m.E = pyo.Set(initialize=E); m.T = pyo.Set(initialize=T)

    m.p_im       = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_ex       = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.b_ch       = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.b_dis      = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.SOC        = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_net      = pyo.Var(m.E, m.T, within=pyo.Reals)
    m.p_up_res   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_down_res = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_up_act   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_down_act = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.a_up       = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.a_down     = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.c_up       = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.c_down     = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.w_up       = pyo.Var(m.T, within=pyo.Binary)
    m.w_down     = pyo.Var(m.T, within=pyo.Binary)
    m.z_up       = pyo.Var(m.T, within=pyo.Binary)
    m.z_down     = pyo.Var(m.T, within=pyo.Binary)

    # (L-1) Linking
    m.c_link_up = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_up_res[e,t] == early_up_acc[e,t] + m.a_up[e,t] - m.c_up[e,t])
    m.c_link_dn = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_down_res[e,t] == early_down_acc[e,t] + m.a_down[e,t] - m.c_down[e,t])

    # (L-obj)
    def obj_rule(m):
        return sum(
            buy_price[t]  * (sum(m.p_im[e,t] for e in E) - sum(m.p_down_act[e,t] for e in E))
          - sell_price[t] * (sum(m.p_ex[e,t] for e in E) - sum(m.p_up_act[e,t]   for e in E))
          - fcrd_up_early_price[t]   * sum(early_up_acc[e,t]  for e in E)
          - fcrd_down_early_price[t] * sum(early_down_acc[e,t] for e in E)
          + buyback_up_price[t]   * sum(m.c_up[e,t]   for e in E)
          + buyback_down_price[t] * sum(m.c_down[e,t] for e in E)
          - fcrd_up_late_price[t]   * sum(m.a_up[e,t]   for e in E)
          - fcrd_down_late_price[t] * sum(m.a_down[e,t] for e in E)
          for t in T)
    m.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

    # Base constraints (same structure as early, with per-EC params)
    m.c_net = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_net[e,t] - m.p_im[e,t] + m.p_ex[e,t] == 0)
    m.c_pb = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_net[e,t] + pvs[e,t] - loads[e,t]
            - (m.b_ch[e,t] + m.p_down_act[e,t])
            + (m.b_dis[e,t] + m.p_up_act[e,t]) == 0)
    m.c_act_up = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_up_act[e,t] == y_acc_up[t]*act_frac_up[t]*m.p_up_res[e,t])
    m.c_act_dn = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_down_act[e,t] == y_acc_down[t]*act_frac_down[t]*m.p_down_res[e,t])

    m.c_soc0 = pyo.Constraint(m.E,
        rule=lambda m,e: m.SOC[e,0] == SOC_INIT_FRAC * s_max_arr[e]
            + eta_arr[e]*(m.b_ch[e,0]+m.p_down_act[e,0])
            - (1.0/eta_arr[e])*(m.b_dis[e,0]+m.p_up_act[e,0]))
    def soc_dyn(m,e,t):
        if t==0: return pyo.Constraint.Skip
        return m.SOC[e,t] == m.SOC[e,t-1] \
            + eta_arr[e]*(m.b_ch[e,t]+m.p_down_act[e,t]) \
            - (1.0/eta_arr[e])*(m.b_dis[e,t]+m.p_up_act[e,t])
    m.c_soc_dyn = pyo.Constraint(m.E, m.T, rule=soc_dyn)
    m.c_soc_end = pyo.Constraint(m.E,
        rule=lambda m,e: m.SOC[e, T_MAX-1] == SOC_INIT_FRAC * s_max_arr[e])

    m.c_sus_up = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.SOC[e,t] >= y_acc_up[t]*T_SUSTAIN/eta_arr[e]*m.p_up_res[e,t])
    m.c_sus_dn = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: s_max_arr[e] - m.SOC[e,t] >= y_acc_down[t]*T_SUSTAIN*eta_arr[e]*m.p_down_res[e,t])

    m.c_dis_lim = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.b_dis[e,t] + m.p_up_res[e,t] + UPSILON_P*m.p_down_res[e,t] <= b_max_arr[e])
    m.c_ch_lim = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.b_ch[e,t] + m.p_down_res[e,t] + UPSILON_P*m.p_up_res[e,t] <= b_max_arr[e])
    m.c_soc_cap = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.SOC[e,t] <= s_max_arr[e])

    # Late-specific constraints
    m.c_tu_up_lo = pyo.Constraint(m.T, rule=lambda m,t: p_min*m.w_up[t] <= sum(m.a_up[e,t] for e in E))
    m.c_tu_up_hi = pyo.Constraint(m.T, rule=lambda m,t: sum(m.a_up[e,t] for e in E) <= p_max*m.w_up[t])
    m.c_tu_dn_lo = pyo.Constraint(m.T, rule=lambda m,t: p_min*m.w_down[t] <= sum(m.a_down[e,t] for e in E))
    m.c_tu_dn_hi = pyo.Constraint(m.T, rule=lambda m,t: sum(m.a_down[e,t] for e in E) <= p_max*m.w_down[t])
    m.c_fin_up_lo = pyo.Constraint(m.T, rule=lambda m,t: p_min*m.z_up[t] <= sum(m.p_up_res[e,t] for e in E))
    m.c_fin_up_hi = pyo.Constraint(m.T, rule=lambda m,t: sum(m.p_up_res[e,t] for e in E) <= p_max*m.z_up[t])
    m.c_fin_dn_lo = pyo.Constraint(m.T, rule=lambda m,t: p_min*m.z_down[t] <= sum(m.p_down_res[e,t] for e in E))
    m.c_fin_dn_hi = pyo.Constraint(m.T, rule=lambda m,t: sum(m.p_down_res[e,t] for e in E) <= p_max*m.z_down[t])
    m.c_nosim_up = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.c_up[e,t] <= early_up_acc[e,t]*(1-m.w_up[t]))
    m.c_nosim_dn = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.c_down[e,t] <= early_down_acc[e,t]*(1-m.w_down[t]))
    return m

print('build_late_model() defined')


## 6. Solver & day pipeline

In [ ]:
def solve_model(model):
    res = SolverFactory(SOLVER_NAME).solve(model, tee=False)
    tc = res.solver.termination_condition
    if tc not in (pyo.TerminationCondition.optimal, pyo.TerminationCondition.feasible):
        raise RuntimeError(f'Solver: {tc}')
    return res

def xvar2d(var, n, t=T_MAX):
    a = np.zeros((n, t))
    for e in range(n):
        for h in range(t):
            a[e,h] = pyo.value(var[e,h])
    return a

def solve_day(date_val, df_day, loads, pvs, cfg, run_base=True):
    n_ec = cfg['n_ec']
    b_max_arr = cfg['b_max']; s_max_arr = cfg['s_max']; eta_arr = cfg['eta']
    p_min = cfg['p_min']; p_max = cfg['p_max']

    buy  = df_day[COL_BUY].values;  sell = df_day[COL_SELL].values
    up_e = df_day[COL_UP_EARLY].values; dn_e = df_day[COL_DOWN_EARLY].values
    up_l = df_day[COL_UP_LATE].values;  dn_l = df_day[COL_DOWN_LATE].values
    af_u = df_day[COL_ACT_UP].values;   af_d = df_day[COL_ACT_DOWN].values
    ya_u = df_day['y_acc_up'].values;   ya_d = df_day['y_acc_down'].values
    bb_u = df_day['buyback_up'].values; bb_d = df_day['buyback_down'].values

    # Early
    me = build_early_model(loads, pvs, buy, sell, up_e, dn_e, af_u, af_d, ya_u, ya_d,
                           b_max_arr, s_max_arr, eta_arr, p_min, p_max)
    solve_model(me)
    early_up = xvar2d(me.p_up_res, n_ec)
    early_dn = xvar2d(me.p_down_res, n_ec)
    early_up_acc = early_up.copy(); early_dn_acc = early_dn.copy()

    # Late
    ml = build_late_model(loads, pvs, buy, sell, up_e, dn_e, up_l, dn_l, bb_u, bb_d,
                          af_u, af_d, ya_u, ya_d, early_up_acc, early_dn_acc,
                          b_max_arr, s_max_arr, eta_arr, p_min, p_max)
    solve_model(ml)

    # Extract
    fin_up  = xvar2d(ml.p_up_res, n_ec);  fin_dn  = xvar2d(ml.p_down_res, n_ec)
    fin_im  = xvar2d(ml.p_im, n_ec);      fin_ex  = xvar2d(ml.p_ex, n_ec)
    fin_soc = xvar2d(ml.SOC, n_ec)
    fin_bch = xvar2d(ml.b_ch, n_ec);      fin_bdis = xvar2d(ml.b_dis, n_ec)
    tu_up = xvar2d(ml.a_up, n_ec); tu_dn = xvar2d(ml.a_down, n_ec)
    cn_up = xvar2d(ml.c_up, n_ec); cn_dn = xvar2d(ml.c_down, n_ec)

    A = lambda x: x.sum(axis=0)

    import_cost = (buy * (A(fin_im) - A(xvar2d(ml.p_down_act, n_ec)))).sum()
    export_rev  = (sell * (A(fin_ex) - A(xvar2d(ml.p_up_act, n_ec)))).sum()
    fcrd_up_early_rev = (up_e * A(early_up_acc)).sum()
    fcrd_dn_early_rev = (dn_e * A(early_dn_acc)).sum()
    fcrd_up_late_rev  = (up_l * A(tu_up)).sum()
    fcrd_dn_late_rev  = (dn_l * A(tu_dn)).sum()
    buyback_up_cost   = (bb_u * A(cn_up)).sum()
    buyback_dn_cost   = (bb_d * A(cn_dn)).sum()

    r = {
        'date': date_val,
        'net_energy_cost': import_cost - export_rev,
        'import_cost': import_cost, 'export_rev': export_rev,
        'fcrd_up_early_rev': fcrd_up_early_rev, 'fcrd_dn_early_rev': fcrd_dn_early_rev,
        'fcrd_up_late_rev': fcrd_up_late_rev,   'fcrd_dn_late_rev': fcrd_dn_late_rev,
        'buyback_up_cost': buyback_up_cost,     'buyback_dn_cost': buyback_dn_cost,
        'fcrd_up_net': fcrd_up_early_rev + fcrd_up_late_rev - buyback_up_cost,
        'fcrd_dn_net': fcrd_dn_early_rev + fcrd_dn_late_rev - buyback_dn_cost,
    }
    r['total_fcrd_rev'] = r['fcrd_up_net'] + r['fcrd_dn_net']

    # Base case (battery only, no FCR-D)
    if run_base:
        z24 = np.zeros(T_MAX)
        # Base model: no FCR-D → set prices and acceptance to 0, use full battery power limits
        mb = build_early_model(loads, pvs, buy, sell, z24, z24, af_u, af_d, z24, z24,
                               b_max_arr, s_max_arr, eta_arr, p_min, p_max)
        solve_model(mb)
        base_im = xvar2d(mb.p_im, n_ec).sum(axis=0)
        base_ex = xvar2d(mb.p_ex, n_ec).sum(axis=0)
        r['base_energy_cost'] = (buy * base_im - sell * base_ex).sum()
        net_load = loads.sum(axis=0) - pvs.sum(axis=0)
        r['no_battery_cost'] = (buy * np.maximum(net_load, 0) - sell * np.maximum(-net_load, 0)).sum()

    return r

print('solve_day() defined')


## 7. Run all portfolio configurations (full year 2025)

In [ ]:
day_counts = df_all.groupby('date').size()
complete_dates = sorted([d for d, c in day_counts.items() if c == 24])
print(f'Complete 24h days: {len(complete_dates)}')

all_results = {}

for ci, cfg in enumerate(ALL_CONFIGS):
    name = cfg['name']
    print(f'\n{"="*60}')
    print(f'Config {ci+1}/{len(ALL_CONFIGS)}: {name}')
    print(f'  {cfg["n_ec"]} ECs  |  {cfg["p_max"]:.0f} kW / {cfg["s_total"]:.0f} kWh')
    print(f'{"="*60}')

    # Generate portfolio (same seed → same type assignment,
    # but we take n_ec entries from the sequence)
    ec_types, scale_factors = generate_portfolio(
        cfg['n_ec'], PROB_B_TYPE, SCALE_MU, SCALE_SIGMA, RNG_SEED)

    results_list = []
    failed = []
    t0 = time.time()

    for i, dv in enumerate(complete_dates):
        try:
            dd = df_all[df_all['date'] == dv].sort_values('hour_utc').reset_index(drop=True)
            ld, pv = get_ec_profiles(dd, ec_types, scale_factors)
            r = solve_day(dv, dd, ld, pv, cfg, run_base=True)
            results_list.append(r)
            if (i+1) % 60 == 0 or (i+1) == len(complete_dates):
                print(f'  [{i+1:>3}/{len(complete_dates)}] {dv}  '
                      f'FCRD={r["total_fcrd_rev"]:>8.1f}  elapsed={time.time()-t0:.0f}s')
        except Exception as ex:
            failed.append((dv, str(ex)))
            if len(failed) <= 3:
                print(f'  FAILED {dv}: {ex}')

    print(f'  Done: {len(results_list)} solved, {len(failed)} failed in {time.time()-t0:.0f}s')
    all_results[name] = results_list


## 8. Results summary & CSV export

In [ ]:
scalar_keys = ['date','net_energy_cost','import_cost','export_rev',
               'fcrd_up_early_rev','fcrd_dn_early_rev',
               'fcrd_up_late_rev','fcrd_dn_late_rev',
               'buyback_up_cost','buyback_dn_cost',
               'fcrd_up_net','fcrd_dn_net','total_fcrd_rev',
               'base_energy_cost','no_battery_cost']

rows = []
for cfg in ALL_CONFIGS:
    name = cfg['name']
    for r in all_results.get(name, []):
        row = {k: r[k] for k in scalar_keys if k in r}
        row['config'] = name
        row['n_ec'] = cfg['n_ec']
        row['p_max_kw'] = cfg['p_max']
        row['s_total_kwh'] = cfg['s_total']
        row['eta_min'] = float(cfg['eta'].min())
        row['eta_max'] = float(cfg['eta'].max())
        rows.append(row)

df_res = pd.DataFrame(rows)
df_res['date'] = pd.to_datetime(df_res['date'])
df_res['month'] = df_res['date'].dt.month
df_res['batt_arb_saving']      = df_res['no_battery_cost'] - df_res['base_energy_cost']
df_res['arb_improvement_fcrd'] = df_res['base_energy_cost'] - df_res['net_energy_cost']
df_res['total_value']          = df_res['batt_arb_saving'] + df_res['arb_improvement_fcrd'] + df_res['total_fcrd_rev']

# ── Summary table ────────────────────────────────────────────────────────
print(f'\n{"="*80}')
print(f'{"Config":<25} {"kW":>6} {"kWh":>7} {"Arb TDKK":>9} {"ΔArb":>7} '
      f'{"Up TDKK":>8} {"Dn TDKK":>8} {"Total TDKK":>11} {"FCR-D%":>7}')
print(f'{"-"*80}')

for cfg in ALL_CONFIGS:
    name = cfg['name']
    d = df_res[df_res['config'] == name]
    if len(d) == 0: continue
    arb  = d['batt_arb_saving'].sum() / 1000
    darb = d['arb_improvement_fcrd'].sum() / 1000
    up   = d['fcrd_up_net'].sum() / 1000
    dn   = d['fcrd_dn_net'].sum() / 1000
    tot  = d['total_value'].sum() / 1000
    fshare = (up + dn) / tot * 100 if tot > 0 else 0
    print(f'{name:<25} {cfg["p_max"]:>6.0f} {cfg["s_total"]:>7.0f} '
          f'{arb:>9.1f} {darb:>7.1f} {up:>8.1f} {dn:>8.1f} {tot:>11.1f} {fshare:>6.1f}%')
print(f'{"="*80}')

# ── Save to CSV ──────────────────────────────────────────────────────────
out_path = os.path.join(DATA_OUT_DIR, 'portfolio_study_2025.csv')
df_res.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}  ({len(df_res)} rows)')
